# TranslateMe: Custom Transformer Architecture Deep Dive
### Step-by-Step Implementation and Mathematical Exploration (Vaswani et al., 2017)

This notebook demonstrates the internal mathematical components of our from-scratch PyTorch Transformer:
1. **Sinusoidal Positional Encoding**
2. **Scaled Dot-Product Attention & Causal Masking**
3. **Multi-Head Attention Projections**
4. **Full Seq2SeqTransformer Forward Pass**

In [ ]:
import sys
sys.path.append('..')
import torch
from ml_lab.custom_transformer import (
    PositionalEncoding,
    ScaledDotProductAttention,
    MultiHeadAttention,
    Seq2SeqTransformer,
    TransformerConfig
)
print('PyTorch Version:', torch.__version__)
print('Device:', 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. Sinusoidal Positional Encoding
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
pe = PositionalEncoding(d_model=256, max_len=100, dropout=0.0)
x = torch.zeros(1, 100, 256)
pe_output = pe(x)
print('Positional Encoding Tensor Shape:', pe_output.shape)
assert pe_output.shape == (1, 100, 256)

## 2. Scaled Dot-Product Attention
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

In [ ]:
attn_module = ScaledDotProductAttention(dropout=0.0)
q = torch.randn(1, 4, 10, 64)
k = torch.randn(1, 4, 10, 64)
v = torch.randn(1, 4, 10, 64)
out, weights = attn_module(q, k, v)
print('Attention output shape:', out.shape)
print('Attention weights shape:', weights.shape)
assert out.shape == (1, 4, 10, 64)
assert weights.shape == (1, 4, 10, 10)

## 3. Custom Seq2Seq Transformer Model Initialization
Instantiating our 6.03M parameter configuration from random initialization.

In [ ]:
config = TransformerConfig.small()
model = Seq2SeqTransformer(
    src_vocab_size=2000,
    tgt_vocab_size=2000,
    d_model=config.d_model,
    n_heads=config.n_heads,
    num_encoder_layers=config.num_encoder_layers,
    num_decoder_layers=config.num_decoder_layers,
    d_ff=config.d_ff,
    dropout=config.dropout,
    share_embeddings=True
)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
assert total_params == 6033408